In [1]:
from pathlib import Path
p = Path("api/main.py")
s = p.read_text(encoding="utf-8")
s = s.replace("@localhost:5432/gis_refresh", "@{os.environ.get('PGHOST', 'localhost')}:5432/gis_refresh")
p.write_text(s, encoding="utf-8")
print("PGHOST" in s)

True


In [2]:
%%writefile api/requirements.txt
fastapi
uvicorn
sqlalchemy
psycopg2-binary

Writing api/requirements.txt


In [3]:
%%writefile Dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY api/requirements.txt api/requirements.txt
RUN pip install --no-cache-dir -r api/requirements.txt
COPY api api
COPY web web
EXPOSE 8000
CMD ["uvicorn", "main:app", "--app-dir", "api", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile


In [4]:
%%writefile .dockerignore
.git
sample-data
*.ipynb
.ipynb_checkpoints
__pycache__
tests

Writing .dockerignore


In [5]:
%%writefile .github/workflows/tests.yml
name: tests
on:
  push:
  pull_request:
jobs:
  pytest:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install geopandas rasterio rasterstats shapely numpy sqlalchemy fastapi httpx pytest
      - run: python -m pytest -q
  docker:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: docker build -t region-api .
      - run: docker run -d -p 8000:8000 --name api region-api
      - run: |
          for i in $(seq 1 20); do
            curl -fs http://127.0.0.1:8000/health && exit 0
            sleep 1
          done
          docker logs api
          exit 1
      - run: curl -fs http://127.0.0.1:8000/map/ | grep -i maplibre

Overwriting .github/workflows/tests.yml


In [6]:
%%writefile -a README.md

## Docker

`docker build -t region-api .` builds the API image. Run it with `docker run -p 8000:8000 -e PGPASSWORD=... -e PGHOST=<database host> region-api`. The CI workflow builds the image and checks `/health` and `/map/` on every push.

Appending to README.md
